# Camera population figure trial

Eight existing 18 September stacks were found locally. Each has 150 acquired frames plus a phase marker, and its metadata explicitly says vibration **off** and static background **already subtracted**. These are exploratory displays, not the final controlled hinge-yield experiment. No association with Hugh's August APD rods is inferred.

This trial retains the source's multiscale stationary-signal detector (40 evenly sampled detection frames, rather than 80, to bound memory). Detection is polarization-sensitive and brightness-gated. Large blobs and broad angular distributions are retained and flagged in the CSV, not called confirmed single rods. There is no inferred percentage of correctly assembled hinges. FOVs may overlap; summed counts are not guaranteed to be unique molecules.

## Changes from the original survey

1. Use the explicit annular optical model, with Fresnel comparison available through `paper_figures.MODEL`.
2. Reject rather than clip out-of-range radii; keep flags and counts. A near-zero radius is a noisy estimate near the pole, not mathematically undefined θ; φ is undefined at exactly zero.
3. Read parity `[90,45;135,0]` and ROI offsets from metadata. This trial rejects odd-offset stacks rather than silently shifting the convention.
4. Use equal numbers of real frames (150), excluding the appended marker by metadata. One orientation per object comes from **mean channel intensities**; the alternative time cloud gives every accepted object total weight one, even when valid frame counts differ.
5. Correct the optional background annulus to lie outside the signal crop, separately by channel. It is **off by default** because these files are already background-subtracted. Enabling a second subtraction needs a reason and a sensitivity comparison; it cannot remove coherent interference.
6. Replace ordinary per-degree histogram alone with stereographic probability per steradian and a solid-angle corrected θ profile. A common φ cut is used across all FOVs.

Read `camera_spot_summary.csv` for all bright candidates and rejection/size/angle flags. The teal reference is optical θ=40° from the emails, not a calibrated DNA opening angle.

In [ ]:
%matplotlib inline
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
import camera_trial as ct
import paper_figures as pf
RECOMPUTE = False  # True reprocesses all 8 existing full stacks; never edits them
ct.LOCAL_BACKGROUND=False
ct.FRAMES=150
if RECOMPUTE:
    camera_results=ct.run_camera()
print((pf.ROOT/'camera_summary.json').read_text())


In [ ]:
display(Image(filename=str(pf.OUT/'camera_population_stereographic.png')))
display(Image(filename=str(pf.OUT/'camera_theta_profiles.png')))

## Equal-rod-weight time-cloud alternative

Each rod contributes its per-frame orientations, normalized by its own valid-frame count. Thus a long or less rejected trajectory cannot dominate the population. This is distinct from the one-mean-channel-orientation map, because averaging and nonlinear angle inversion do not commute.

In [ ]:
from matplotlib.colors import LogNorm
paths=sorted(pf.ROOT.glob('frame_stack_*_population.npz'))
fig,axs=plt.subplots(2,4,figsize=(12,6),layout='constrained')
data=[dict(np.load(path)) for path in paths]
norm=LogNorm(.01,max(d['time_density'].max() for d in data))
for ax,path,d in zip(axs.flat,paths,data):
    z=d['time_density'];m=ax.pcolormesh(d['ex'],d['ey'],np.ma.masked_where(z.T<=0,z.T),cmap='magma',norm=norm,rasterized=True)
    ax.set(aspect='equal',title=path.stem[21:27],xlabel='u',ylabel='v')
fig.colorbar(m,ax=axs,label='equal-rod-weight density (sr⁻¹)',shrink=.7)
fig.suptitle('Vibration off · exploratory time-cloud distributions')
pf.save(fig,'camera_equal_rod_time_clouds');plt.show()


## Data needed for the final DNA figure

Hugh's original **matched Mode 1 + Mode 2 hinge recordings**, including the chosen rod coordinates/IDs, raw four-channel information, timing, background and vibration settings, are needed to reproduce Figure 5c–d and establish the screening-to-follow-up link. A static correctly oriented rod from the same preparation would supply the benchmark Richard requested. The APD files cannot reconstruct those camera panels.

For the later measurements: matched preparation comparisons, vibration-on movies and a no-DNA control; static rod examples at the target optical angle; short exposures or APDs to bound rapid motion. The September files do not supply those controls. Retain all detected objects and selection counts so gating does not masquerade as improved yield.